In [21]:
#Extracts all movements of voz units given a combined file of data from multiple units

import os
import sys

sys.path.append(os.path.abspath(".."))
from data_tools import extract_movements
from data_tools import my_setup
from data_tools import deployment_sets


file_path = my_setup.raw_2025data_path()
device_names = deployment_sets.devices_2025()
data_file = rf"{file_path}combined_data.csv"
unit_locations = rf"{file_path}unit_locations.csv"

### Uncomment if file is not already created
# extract_movements.from_combined(data_file, device_names, unit_locations, decimal_threshold=0.05)

In [52]:
import pandas as pd
import folium
from folium.features import DivIcon

### Choose measurement type
# measurement_type = "ozone"
measurement_type = "pm25"

CARB_locations = rf"../reference_files/FEMCoordinates{measurement_type}.csv"
sensor = "McFarland"

# Read datasets
df_VOZ = pd.read_csv(unit_locations)
df_CARB = pd.read_csv(CARB_locations)

# Filter VOZ unit for this sensor
unit_df = df_VOZ[df_VOZ['name'] == sensor].reset_index(drop=True)

# Create base map
m = folium.Map(location=[36.7783, -119.4179], zoom_start=9, control_scale=True)

# Custom flag icon for VOZ
flag_icon = folium.CustomIcon(
    icon_image="../reference_files/flag.png",
    icon_size=(32, 32),
    icon_anchor=(16, 32)  # bottom-center of the flag
)

# Keep track of text positions to avoid overlap
textlocations = []
text_offset = 0.0015  # latitude offset for stacking

# --- Plot VOZ unit markers with date labels ---
for i, row in enumerate(unit_df.itertuples()):
    location = [row.latitude, row.longitude]

    # Plot flag
    folium.Marker(
        location=location,
        icon=flag_icon
    ).add_to(m)

    # Prepare label text
    ts = getattr(row, 'timestamp')
    date_text = ts.strftime("%m/%d") if hasattr(ts, "strftime") else "/".join(str(ts).split("/")[:2])

    if i == 0:
        label_text = f"Start {date_text}"
    elif i == len(unit_df) - 1:
        label_text = f"End {date_text}"
    else:
        label_text = date_text

    # Initial label position slightly above the flag
    label_lat = location[0] + 0.03  # small offset above flag
    label_lon = location[1] + 0.08

    # Stack labels downward if they would overlap previous labels
    overlap = True
    while overlap:
        overlap = False
        for prev_lat, prev_lon in textlocations:
            if abs(label_lon - prev_lon) < 0.0001 and abs(label_lat - prev_lat) < text_offset:
                label_lat = prev_lat - 0.04 # move downward
                overlap = True

    textlocations.append([label_lat, label_lon])

    # Add label above flag using icon_anchor to position bottom-center
    folium.Marker(
        location=[label_lat, label_lon],
        icon=DivIcon(
            icon_size=(100, 36),
            icon_anchor=(50, 36),  # bottom-center of div
            html=f"""
            <div style="
                font-size:16px;
                font-weight:bold;
                font-family: 'Times New Roman', serif;
                color:black;
                text-align:center;
                white-space: nowrap;
            ">
                {label_text}
            </div>
            """
        )
    ).add_to(m)

# --- Plot CARB monitors ---
for _, row in df_CARB.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        icon=folium.DivIcon(html="""<div style="font-size: 28px;">🔷</div>"""),
        popup=row['name']
    ).add_to(m)

# --- Add legend ---
legend_html = '''
<div style="
     position: fixed; 
     top: 50px; right: 50px; width: 160px; height: 110px; 
     background-color: white;
     border:2px solid grey; 
     z-index:9999; 
     font-size:14px;
     padding: 10px;
     box-shadow: 2px 2px 4px rgba(0,0,0,0.3);
     ">
  <b>Legend</b><br><br>
  🔷 CARB Monitors<br>
  🚩 VOZ Monitors<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save('map.html')
